# Computer Vision Workshop: Building Monkey Thinking

Welcome! In this workshop, we'll build a hand gesture detection app step-by-step. By the end, you'll have written the complete Monkey Thinking application that detects your hand gestures and displays monkey memes in real-time.

## What We're Building

An app that:
- Detects hands using your webcam
- Recognizes the "pointing" gesture
- Shows green dots on your hand landmarks
- Displays monkey memes that change based on your gesture

## Workshop Structure

1. Part 1: Detecting Hand Landmarks
2. Part 2: Working with Images and Colors
3. Part 3: Building the Complete Application

Let's get started!

## Part 1: Understanding the Technologies

### What is Computer Vision?

Computer vision is a field of artificial intelligence that enables computers to interpret and understand visual information from the world. It allows machines to "see" and make decisions based on images or video.

### Key Technologies We'll Use

1. **OpenCV (cv2)**: The most popular computer vision library for real-time image and video processing
2. **MediaPipe**: Google's framework for building machine learning pipelines for video/audio processing
3. **NumPy**: Python's fundamental package for numerical computing

### How Hand Tracking Works

MediaPipe uses a machine learning model trained on thousands of hand images to:
1. Detect hands in video frames
2. Identify 21 landmarks (key points) on each hand
3. Track these landmarks across frames in real-time

The 21 landmarks include:
- Wrist (0)
- Thumb: 1-4
- Index finger: 5-8
- Middle finger: 9-12
- Ring finger: 13-16
- Pinky: 17-20

## Part 2: Setting Up the Environment

First, let's install the required libraries. Make sure you have Python 3.8 or higher installed.

In [ ]:
# Install required packages
# Run this cell only once
# !pip install opencv-python mediapipe numpy

Let's verify the installations and import our libraries:

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print(f"OpenCV version: {cv2.__version__}")
print(f"MediaPipe version: {mp.__version__}")
print(f"NumPy version: {np.__version__}")
print("\nAll libraries imported successfully!")

## Part 3: Loading the Hand Tracking Model

MediaPipe provides pre-trained models for hand detection. We'll use the HandLandmarker model which detects and tracks hand landmarks.

In [ ]:
# Configure the hand landmarker model
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')

# Set options for hand detection
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2,  # Can detect up to 2 hands
    running_mode=vision.RunningMode.VIDEO,  # For video stream processing
    min_hand_detection_confidence=0.6,  # Minimum confidence threshold (60%)
    min_tracking_confidence=0.6  # Tracking confidence threshold
)

# Create the landmarker
landmarker = vision.HandLandmarker.create_from_options(options)

print("Hand landmarker model loaded successfully!")

### Understanding the Configuration

- **num_hands**: Maximum number of hands to detect (we set it to 2)
- **running_mode**: VIDEO mode for processing video streams
- **min_hand_detection_confidence**: How confident the model should be before detecting a hand (0.0-1.0)
- **min_tracking_confidence**: Confidence threshold for tracking hands across frames

## Part 4: Loading Meme Images

We'll load different meme images that will be displayed based on the detected gestures.

In [ ]:
# Load meme images
meme_staring = cv2.imread("meme/staring.png")
meme_pointing = cv2.imread("meme/pointing.png")
meme_thinking = cv2.imread("meme/thinking.png")

# Check if images loaded successfully
if meme_staring is None or meme_pointing is None:
    print("Error: Could not load meme images. Make sure the 'meme' folder exists!")
else:
    print("Meme images loaded successfully!")
    print(f"Staring meme size: {meme_staring.shape}")
    print(f"Pointing meme size: {meme_pointing.shape}")

## Part 5: Understanding Hand Landmarks

Each hand has 21 landmarks. Let's understand the key landmarks we'll use for gesture detection:

In [ ]:
# Hand landmark indices
WRIST = 0

# Thumb
THUMB_CMC = 1
THUMB_MCP = 2
THUMB_IP = 3
THUMB_TIP = 4

# Index finger
INDEX_FINGER_MCP = 5
INDEX_FINGER_PIP = 6
INDEX_FINGER_DIP = 7
INDEX_FINGER_TIP = 8

# Middle finger
MIDDLE_FINGER_MCP = 9
MIDDLE_FINGER_PIP = 10
MIDDLE_FINGER_DIP = 11
MIDDLE_FINGER_TIP = 12

# Ring finger
RING_FINGER_MCP = 13
RING_FINGER_PIP = 14
RING_FINGER_DIP = 15
RING_FINGER_TIP = 16

# Pinky
PINKY_MCP = 17
PINKY_PIP = 18
PINKY_DIP = 19
PINKY_TIP = 20

print("Landmark indices defined!")
print("\nFor our pointing gesture, we'll use:")
print(f"- Index tip: {INDEX_FINGER_TIP}")
print(f"- Index PIP: {INDEX_FINGER_PIP}")
print(f"- Middle tip: {MIDDLE_FINGER_TIP}")
print(f"- Middle PIP: {MIDDLE_FINGER_PIP}")

## Part 6: Gesture Recognition Logic

Now let's understand how we detect a "pointing" gesture. A pointing gesture typically means:
- Index finger is extended (pointing up/forward)
- Other fingers (middle, ring, pinky) are folded down

We determine if a finger is extended by comparing the Y-coordinate of the fingertip with its PIP joint:
- If fingertip.y < PIP.y, the finger is extended (pointing up)
- If fingertip.y > PIP.y, the finger is folded down

Note: In image coordinates, Y=0 is at the top, so lower Y values mean higher position.

In [ ]:
def detect_pointing_gesture(hand_landmarks):
    """
    Detect if the hand is making a pointing gesture.
    
    Args:
        hand_landmarks: List of 21 hand landmark points
        
    Returns:
        bool: True if pointing gesture detected, False otherwise
    """
    # Get relevant landmarks
    index_tip = hand_landmarks[INDEX_FINGER_TIP]
    index_pip = hand_landmarks[INDEX_FINGER_PIP]
    middle_tip = hand_landmarks[MIDDLE_FINGER_TIP]
    middle_pip = hand_landmarks[MIDDLE_FINGER_PIP]
    ring_tip = hand_landmarks[RING_FINGER_TIP]
    ring_pip = hand_landmarks[RING_FINGER_PIP]
    pinky_tip = hand_landmarks[PINKY_TIP]
    pinky_pip = hand_landmarks[PINKY_PIP]
    
    # Check finger states
    index_extended = index_tip.y < index_pip.y
    middle_folded = middle_tip.y > middle_pip.y
    ring_folded = ring_tip.y > ring_pip.y
    pinky_folded = pinky_tip.y > pinky_pip.y
    
    # Pointing gesture: only index finger extended
    if index_extended and middle_folded and ring_folded and pinky_folded:
        return True
    return False

print("Gesture detection function defined!")

## Part 7: The Main Video Processing Loop

Now we'll create the main application that:
1. Captures video from your webcam
2. Processes each frame to detect hands
3. Recognizes gestures
4. Displays the appropriate meme
5. Shows visual feedback with landmarks

This is the heart of our application!

In [ ]:
# This is a complete working version of the application
# You can run this cell to start the application

def run_monkey_thinking():
    """
    Main function to run the Monkey Thinking application.
    Press 'q' to quit.
    """
    # Initialize video capture
    cap = cv2.VideoCapture(1)  # Try 0 if 1 doesn't work
    timestamp = 0
    
    # Check if camera opened successfully
    if not cap.isOpened():
        print("Error: Could not open camera. Try changing the camera index.")
        return
    
    print("Camera opened successfully!")
    print(f"Camera resolution: {int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}")
    print("Press 'q' to quit")
    
    # Set initial meme
    current_meme = cv2.imread("meme/staring.png")
    
    while cap.isOpened():
        # Read frame from camera
        valid, frame = cap.read()
        
        if not valid:
            print("Warning: Could not read frame")
            break
        
        # Convert BGR to RGB (MediaPipe uses RGB)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Create MediaPipe image
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )
        
        # Detect hands
        result = landmarker.detect_for_video(mp_image, timestamp)
        timestamp += 1
        
        # Process detected hands
        if result.hand_landmarks:
            for hand in result.hand_landmarks:
                # Detect pointing gesture
                if detect_pointing_gesture(hand):
                    current_meme = cv2.imread("meme/pointing.png")
                else:
                    current_meme = cv2.imread("meme/staring.png")
                
                # Draw landmarks on the frame
                for lm in hand:
                    h, w, _ = frame.shape
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)
        else:
            # No hands detected
            current_meme = cv2.imread("meme/staring.png")
        
        # Combine meme and camera frame side by side
        frame_height, frame_width = frame.shape[:2]
        meme_resized = cv2.resize(current_meme, (frame_width, frame_height))
        combined = np.hstack([meme_resized, frame])
        
        # Display the result
        cv2.imshow('Think Monke', combined)
        
        # Check for quit key
        if cv2.waitKey(5) & 0xFF == ord('q'):
            break
    
    # Cleanup
    cap.release()
    cv2.destroyAllWindows()
    landmarker.close()
    print("Application closed successfully!")

# Uncomment the line below to run the application
# run_monkey_thinking()

## Part 8: Understanding the Video Processing Pipeline

Let's break down what happens in each frame:

### Step 1: Frame Capture
```python
valid, frame = cap.read()
```
Captures a single frame from the webcam.

### Step 2: Color Space Conversion
```python
rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
```
OpenCV captures in BGR format, but MediaPipe expects RGB. We convert the color space.

### Step 3: Hand Detection
```python
result = landmarker.detect_for_video(mp_image, timestamp)
```
MediaPipe processes the frame and returns detected hands with their landmarks.

### Step 4: Gesture Recognition
```python
if detect_pointing_gesture(hand):
    current_meme = cv2.imread("meme/pointing.png")
```
We analyze the hand landmarks to recognize specific gestures.

### Step 5: Visualization
```python
combined = np.hstack([meme_resized, frame])
cv2.imshow('Think Monke', combined)
```
Stack the meme and camera feed horizontally and display them.

## Part 9: Challenges and Extensions

Now that you understand the basics, here are some challenges to extend the project:

### Challenge 1: Add More Gestures

Try adding detection for these gestures:
- Thumbs up
- Peace sign (index + middle finger extended)
- Open palm (all fingers extended)
- Fist (all fingers closed)

### Challenge 2: Add Text Overlays

Display the detected gesture name on the video feed using:
```python
cv2.putText(frame, "Pointing!", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
```

### Challenge 3: Count Gestures

Track how many times each gesture is performed and display statistics.

### Challenge 4: Add Sound Effects

Play different sounds when different gestures are detected using the `pygame` or `playsound` library.

### Challenge 5: Two-Hand Gestures

Detect gestures that require both hands (like clapping or specific patterns).

## Part 10: Exercise - Detect Thumbs Up

Let's implement a thumbs up detector together!

For a thumbs up gesture:
- Thumb should be extended upward
- Other fingers should be folded

Hint: You'll need to check the thumb tip position relative to other joints.

In [ ]:
def detect_thumbs_up(hand_landmarks):
    """
    Exercise: Implement thumbs up detection.
    
    Args:
        hand_landmarks: List of 21 hand landmark points
        
    Returns:
        bool: True if thumbs up detected, False otherwise
    """
    # TODO: Implement thumbs up detection
    # Hint: Check if thumb tip is higher than other finger tips
    # and if other fingers are folded
    
    pass  # Remove this and add your code

# Test your function here

## Part 11: Troubleshooting Common Issues

### Camera Not Opening
- Try changing the camera index from 1 to 0 or 2
- Check if another application is using the camera
- Verify camera permissions in your OS settings

### Poor Detection Accuracy
- Ensure good lighting conditions
- Keep your hand within the camera frame
- Adjust `min_hand_detection_confidence` and `min_tracking_confidence` values

### Slow Performance
- Reduce video resolution
- Process every other frame instead of every frame
- Close other resource-intensive applications

### Images Not Loading
- Verify the `meme` folder exists in the same directory
- Check that image files have correct names and extensions
- Use absolute paths if relative paths don't work

## Part 12: Key Concepts Summary

### Computer Vision Pipeline
1. **Input**: Capture video/images from camera
2. **Processing**: Apply algorithms/models to extract information
3. **Analysis**: Interpret the extracted information
4. **Output**: Display results or trigger actions

### Coordinate Systems
- Image coordinates: (0,0) is top-left corner
- X increases from left to right
- Y increases from top to bottom
- Landmarks are normalized (0.0 to 1.0) relative to frame size

### Real-Time Processing
- Each frame must be processed quickly (typically < 33ms for 30 FPS)
- Balance between accuracy and speed
- Use efficient algorithms and optimized libraries

### Machine Learning in Computer Vision
- Pre-trained models (like MediaPipe) save development time
- Transfer learning allows using models trained on large datasets
- Trade-offs between model size, speed, and accuracy

## Part 13: Additional Resources

### Documentation
- [MediaPipe Hand Landmark Detection](https://developers.google.com/mediapipe/solutions/vision/hand_landmarker)
- [OpenCV Python Tutorials](https://docs.opencv.org/master/d6/d00/tutorial_py_root.html)
- [NumPy Documentation](https://numpy.org/doc/)

### Further Learning
- Explore other MediaPipe solutions (face detection, pose estimation, object detection)
- Learn about deep learning frameworks (TensorFlow, PyTorch)
- Study image processing techniques (filtering, edge detection, morphological operations)
- Understand neural network architectures for computer vision (CNNs)

### Project Ideas
- Build a sign language interpreter
- Create a virtual mouse controlled by hand gestures
- Develop a fitness app that counts exercises
- Make an AR application with hand tracking
- Build a gesture-controlled game

## Conclusion

Congratulations! You've learned how to build a real-time hand gesture detection system using computer vision and machine learning. You now understand:

- How computer vision applications work
- Using pre-trained ML models for hand tracking
- Real-time video processing with OpenCV
- Gesture recognition algorithms
- Building interactive CV applications

Keep experimenting, try the challenges, and most importantly, have fun building amazing computer vision projects!

### Next Steps
1. Complete the exercises in this notebook
2. Experiment with different gestures
3. Customize the application with your own images
4. Share your creation with others
5. Explore other computer vision projects

Thank you for participating in this workshop!